In [17]:
config = Config()

AGG_test = robust_read_csv(config.BASE_PATH + "X_test_AGG.csv")
AGG_train = robust_read_csv(config.BASE_PATH + "X_train_AGG.csv")

W2V_test = robust_read_csv(config.BASE_PATH + "X_test_W2V.csv")
W2V_train = robust_read_csv(config.BASE_PATH + "X_train_W2V.csv")

train_tx = robust_read_csv(config.BASE_PATH + "train_transactions.csv")
test_tx = robust_read_csv(config.BASE_PATH + "test_transactions.csv")

y_train = robust_read_csv(config.BASE_PATH + "y_train.csv")

In [ ]:
W2V_train

,custid,BRD_v001,BRD_v002,BRD_v003,BRD_v004,BRD_v005,BRD_v006,BRD_v007,BRD_v008,BRD_v009,...,D2V_v055,D2V_v056,D2V_v057,D2V_v058,D2V_v059,D2V_v060,D2V_v061,D2V_v062,D2V_v063,D2V_v064
0,0,0.047273,0.062546,0.061084,0.024397,-0.222605,0.107226,-0.058357,0.308702,0.192968,...,-0.010766,0.075378,0.047184,-0.103869,-0.201284,-0.102224,-0.015279,0.285075,-0.068463,0.034374
1,2,0.103754,0.008648,0.112604,0.065722,-0.283691,0.035815,-0.112141,0.259957,0.253435,...,-0.433442,-0.061834,0.194284,-0.158439,0.233558,-0.483934,-0.184548,0.382296,0.175762,0.143086
2,3,0.073731,-0.129830,0.071430,-0.093699,-0.109610,0.064773,-0.109478,0.151212,-0.023027,...,0.093536,-0.081849,0.075760,-0.123480,0.305239,-0.363928,0.126653,0.138756,0.000477,-0.015562
3,4,0.222583,-0.372375,0.071700,0.007047,-0.458353,0.035068,-0.085334,0.106454,-0.029885,...,-0.112586,0.063373,0.027758,-0.063872,0.102580,-0.103652,-0.014945,0.085770,-0.001036,-0.008202
4,5,0.048636,-0.010063,0.088829,0.002697,-0.197597,0.052401,-0.083274,0.168822,0.050011,...,-0.132404,0.122479,-0.052981,0.007123,0.026413,-0.046414,-0.186830,0.334828,0.102297,0.099131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21582,29995,0.097437,-0.010771,0.120191,-0.020777,-0.196384,0.083475,-0.164411,0.122057,0.146216,...,-0.020802,0.982870,0.105482,-1.128041,0.392485,-0.037691,0.455577,-0.291055,-0.568005,-0.440576
21583,29996,0.039639,0.071148,0.193474,0.040375,-0.018339,0.063604,0.049913,0.134847,0.271227,...,-0.599658,0.114310,0.239320,-0.245860,0.572429,-0.629392,-0.310044,0.537891,0.189352,0.092372
21584,29997,0.032144,-0.095233,0.053404,0.079999,-0.193111,0.130526,-0.059057,0.238042,0.147656,...,-0.077226,0.321810,-0.049442,-0.054139,-0.065973,-0.104839,0.040574,-0.102536,-0.137763,-0.014159
21585,29998,0.149703,-0.154583,0.116938,-0.035117,-0.206725,0.177947,-0.047750,0.149507,0.035890,...,-0.110145,0.210905,-0.014617,0.096547,0.137977,0.022813,-0.122551,0.304631,0.207951,-0.019673


In [20]:
def numeric_eda_summary(df: pd.DataFrame, config=None):
    """수치형 컬럼 요약 테이블 생성"""

    if config is None:
        num_cols = train_tx.select_dtypes(include=[np.number]).columns.tolist()
    else:
        drop_cols = [config.COL_ID, config.COL_TARGET] if hasattr(config, 'COL_TARGET') else []
        num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in drop_cols]

    stats = []

    for col in num_cols:
        series = df[col].dropna()

        stats.append({
            "col": col,
            "n_unique": series.nunique(),
            "missing_ratio": df[col].isna().mean(),
            "zero_ratio": (df[col] == 0).mean(),
            "mean": series.mean(),
            "p99": series.quantile(0.99),
            "max": series.max(),
            "skew": series.skew()
        })

    summary = pd.DataFrame(stats)
    summary = summary.sort_values("skew", ascending=False)

    return summary

def categorical_eda_summary(df: pd.DataFrame):
    """범주형 컬럼 요약 테이블 생성"""

    # 수치형 제외하고 object, category 타입만 선택
    cat_cols = train_tx.select_dtypes(include=['object', 'category']).columns.tolist()

    stats = []

    for col in cat_cols:
        series = df[col].astype(str).fillna("MISSING")
        value_counts = series.value_counts(normalize=True)

        stats.append({
            "col": col,
            "n_unique": series.nunique(),
            "missing_ratio": (df[col].isna().mean()),
            "top1": value_counts.index[0],
            "top1_ratio": value_counts.iloc[0],
            "top2": value_counts.index[1] if len(value_counts) > 1 else None,
            "top2_ratio": value_counts.iloc[1] if len(value_counts) > 1 else None,
            "rare_ratio": value_counts[value_counts < 0.01].sum()  # 1% 미만 비중
        })

    summary = pd.DataFrame(stats)
    summary = summary.sort_values("n_unique", ascending=False)

    return summary


In [21]:
num_summary = numeric_eda_summary(train_tx)
cat_summary = categorical_eda_summary(train_tx)

print("=== Numeric Summary ===")
display(num_summary)

print("=== Categorical Summary ===")
display(cat_summary)

=== Numeric Summary ===


,col,n_unique,missing_ratio,zero_ratio,mean,p99,max,skew
7,dis_amt,6693,0.0,0.505287,3.443273e+03,4.590000e+04,1479000,7.350197
8,net_amt,68460,0.0,0.000000,9.569223e+04,9.693340e+05,72000000,6.789737
6,tot_amt,62646,0.0,0.000000,9.913550e+04,1.000000e+06,72000000,6.548129
10,inst_fee,2,0.0,0.971514,2.848577e-02,1.000000e+00,1,5.668749
9,inst_mon,12,0.0,0.000000,1.876234e+00,6.000000e+00,12,2.366602
5,import_flg,2,0.0,0.863137,1.368632e-01,1.000000e+00,1,2.113091
2,sales_day,31,0.0,0.000000,1.469279e+01,3.100000e+01,31,0.179442
1,sales_month,12,0.0,0.000000,1.048402e+01,1.600000e+01,16,0.040213
0,custid,21587,0.0,0.000018,1.496641e+04,2.968300e+04,29999,0.002713
4,goodcd,9750,0.0,0.000000,3.936181e+12,6.318202e+12,8801192410767,-0.039889


=== Categorical Summary ===


,col,n_unique,missing_ratio,top1,top1_ratio,top2,top2_ratio,rare_ratio
2,brd_nm,1848,0.0,식품,0.206788,지오다노,0.016395,0.728686
3,corner_nm,308,0.0,수입종합화장품,0.107910,용기보증,0.082535,0.503136
4,pc_nm,77,0.0,화장품,0.130179,미확인pc,0.062942,0.241500
7,buyer_nm,35,0.0,일반식품,0.206791,화장품,0.135151,0.035194
5,part_nm,30,0.0,공산품,0.089322,명품잡화,0.078996,0.012899
0,sales_dayofweek,7,0.0,일,0.191222,토,0.191202,0.000000
1,str_nm,4,0.0,무역점,0.268674,신촌점,0.264508,0.000000
6,team_nm,4,0.0,잡화가용팀,0.461968,의류패션팀,0.331200,0.000002


In [ ]:
"""
🎯 연령 예측 모델 - 고도화 피처 엔지니어링
==============================================
백화점 소비 패턴 기반 나이 예측을 위한 도메인 특화 피처 생성
"""

import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings('ignore')


class AgeFeatureEngineer:
    """연령 예측을 위한 고급 피처 엔지니어링"""
    
    def __init__(self):
        # 연령대별 선호 브랜드/카테고리 (도메인 지식 기반)
        self.young_brands = ['지오다노', '유니클로', '자라', 'H&M', '탑텐', '스파오', '에잇세컨즈']
        self.middle_brands = ['로가디스', '타임', '마리떼', '엠브로', '빈폴']
        self.senior_brands = ['한섬', '루이까또즈', '파크론', '머렐']
        
    def create_age_specific_features(self, trans_df, agg_df, w2v_df):
        """연령 예측을 위한 특화 피처 생성"""
        print("🎯 연령 예측 특화 피처 생성 시작...")
        
        # 1. 기본 통합 (AGG + W2V)
        base_features = self._merge_base_features(agg_df, w2v_df)
        
        # 2. 거래 기반 피처들
        brand_features = self._create_brand_age_features(trans_df)
        time_features = self._create_time_pattern_features(trans_df)
        category_features = self._create_category_combination_features(trans_df)
        spending_features = self._create_spending_behavior_features(trans_df)
        seasonal_features = self._create_seasonal_response_features(trans_df)
        family_features = self._create_family_composition_features(trans_df)
        digital_features = self._create_digital_affinity_features(trans_df)
        statistical_features = self._create_statistical_features(trans_df)
        
        # 모든 피처 통합
        all_features = [
            base_features,
            brand_features,
            time_features,
            category_features,
            spending_features,
            seasonal_features,
            family_features,
            digital_features,
            statistical_features
        ]
        
        # None 제거하고 통합
        valid_features = [f for f in all_features if f is not None and not f.empty]
        
        if not valid_features:
            print("⚠️ 생성된 피처가 없습니다!")
            return pd.DataFrame()
        
        # custid를 기준으로 outer join
        result = valid_features[0]
        for feat in valid_features[1:]:
            result = result.join(feat, how='outer')
        
        print(f"✅ 총 생성된 피처 수: {result.shape[1]}개")
        return result
    
    def _merge_base_features(self, agg_df, w2v_df):
        """기본 피처 통합"""
        # AGG + W2V 통합
        base = agg_df.merge(w2v_df, on='custid', how='outer')
        return base.set_index('custid')
    
    def _create_brand_age_features(self, df):
        """연령대별 브랜드 선호도 피처"""
        print("  📊 브랜드 연령 선호도 피처 생성...")
        
        if 'brd_nm' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        result = pd.DataFrame(index=df['custid'].unique())
        
        # 브랜드별 구매 금액
        brand_amt = df.groupby(['custid', 'brd_nm'])['tot_amt'].sum().unstack(fill_value=0)
        total_amt = brand_amt.sum(axis=1)
        
        # 연령대별 브랜드 비중
        young_cols = [b for b in self.young_brands if b in brand_amt.columns]
        middle_cols = [b for b in self.middle_brands if b in brand_amt.columns]
        senior_cols = [b for b in self.senior_brands if b in brand_amt.columns]
        
        if young_cols:
            result['brand_young_ratio'] = brand_amt[young_cols].sum(axis=1) / (total_amt + 1e-5)
        if middle_cols:
            result['brand_middle_ratio'] = brand_amt[middle_cols].sum(axis=1) / (total_amt + 1e-5)
        if senior_cols:
            result['brand_senior_ratio'] = brand_amt[senior_cols].sum(axis=1) / (total_amt + 1e-5)
        
        # 브랜드 다양성
        result['brand_diversity'] = df.groupby('custid')['brd_nm'].nunique()
        
        # 브랜드 집중도 (최다 브랜드 비율)
        brand_conc = df.groupby('custid')['brd_nm'].apply(
            lambda x: (x.value_counts().iloc[0] / len(x)) if len(x) > 0 else 0
        )
        result['brand_concentration'] = brand_conc
        
        return result
    
    def _create_time_pattern_features(self, df):
        """시간대 패턴 피처"""
        print("  ⏰ 시간대 패턴 피처 생성...")
        
        if 'sales_time' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        df['hour'] = df['sales_time'] // 100
        
        result = pd.DataFrame(index=df['custid'].unique())
        
        # 주중/주말 구분
        if 'sales_dayofweek' in df.columns:
            weekday_map = {'월': 0, '화': 1, '수': 2, '목': 3, '금': 4, '토': 5, '일': 6}
            df['dow_num'] = pd.to_numeric(df['sales_dayofweek'].map(weekday_map), errors='coerce').fillna(df['sales_dayofweek'])
            df['is_weekday'] = df['dow_num'] <= 4
        
        # 시간대별 패턴
        df['early_morning'] = (df['hour'] < 10).astype(int)
        df['lunch_time'] = df['hour'].between(11, 14).astype(int)
        df['after_school'] = df['hour'].between(16, 19).astype(int)
        df['evening'] = df['hour'].between(19, 21).astype(int)
        df['late_night'] = (df['hour'] >= 21).astype(int)
        
        # 주중 시간대 조합
        if 'is_weekday' in df.columns:
            df['weekday_lunch'] = (df['is_weekday'] & (df['lunch_time'] == 1)).astype(int)
            df['weekday_morning'] = (df['is_weekday'] & (df['early_morning'] == 1)).astype(int)
            df['weekday_afterschool'] = (df['is_weekday'] & (df['after_school'] == 1)).astype(int)
            
            result['weekday_lunch_ratio'] = df.groupby('custid')['weekday_lunch'].mean()
            result['weekday_morning_ratio'] = df.groupby('custid')['weekday_morning'].mean()
            result['weekday_afterschool_ratio'] = df.groupby('custid')['weekday_afterschool'].mean()
        
        # 기본 시간대 비율
        for col in ['early_morning', 'lunch_time', 'after_school', 'evening', 'late_night']:
            result[f'{col}_ratio'] = df.groupby('custid')[col].mean()
        
        # 시간대 변동성
        result['hour_std'] = df.groupby('custid')['hour'].std()
        result['hour_range'] = df.groupby('custid')['hour'].apply(lambda x: x.max() - x.min())
        
        return result
    
    def _create_category_combination_features(self, df):
        """카테고리 조합 패턴"""
        print("  🏷️ 카테고리 조합 피처 생성...")
        
        if 'pc_nm' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        result = pd.DataFrame(index=df['custid'].unique())
        
        # 아동 상품
        df['kids_item'] = df['pc_nm'].str.contains('아동|유아|베이비|키즈', na=False, case=False).astype(int)
        kid_buyers = df[df['kids_item'] == 1]['custid'].unique()
        result['has_kids_signal'] = result.index.isin(kid_buyers).astype(int)
        
        # 명품 + 골프
        df['luxury'] = df['pc_nm'].str.contains('명품|보석|주얼리', na=False, case=False).astype(int)
        df['golf'] = df['pc_nm'].str.contains('골프', na=False, case=False).astype(int)
        
        luxury_golf = df.groupby('custid').apply(
            lambda x: int((x['luxury'].sum() > 0) & (x['golf'].sum() > 0))
        )
        result['luxury_golf_combo'] = luxury_golf
        
        # 화장품 + 패션잡화
        df['cosmetics'] = df['pc_nm'].str.contains('화장품|뷰티|메이크업', na=False, case=False).astype(int)
        df['accessories'] = df['pc_nm'].str.contains('패션잡화|악세사리|핸드백', na=False, case=False).astype(int)
        
        young_female = df.groupby('custid').apply(
            lambda x: int((x['cosmetics'].sum() > 0) & (x['accessories'].sum() > 0))
        )
        result['young_female_combo'] = young_female
        
        # 식품 + 주방용품
        if 'corner_nm' in df.columns:
            df['food'] = df['corner_nm'].str.contains('식품', na=False, case=False).astype(int)
            df['kitchen'] = df['corner_nm'].str.contains('주방|취사', na=False, case=False).astype(int)
            
            housewife = df.groupby('custid').apply(
                lambda x: int((x['food'].sum() > 0) & (x['kitchen'].sum() > 0))
            )
            result['housewife_combo'] = housewife
        
        return result
    
    def _create_spending_behavior_features(self, df):
        """소비 성향 분석 피처"""
        print("  💰 소비 성향 피처 생성...")
        
        if 'tot_amt' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        result = pd.DataFrame(index=df['custid'].unique())
        
        # 가격대별 구매 비중
        df['price_tier'] = pd.cut(
            df['tot_amt'],
            bins=[-np.inf, 50000, 100000, 200000, 500000, np.inf],
            labels=['very_low', 'low', 'mid', 'high', 'premium']
        )
        
        price_dist = pd.crosstab(df['custid'], df['price_tier'], normalize='index')
        for col in price_dist.columns:
            result[f'price_{col}_ratio'] = price_dist[col]
        
        # 객단가 통계
        cust_stats = df.groupby('custid')['tot_amt'].agg(['mean', 'std', 'median'])
        result['ticket_cv'] = cust_stats['std'] / (cust_stats['mean'] + 1e-5)
        
        # Skewness (안전하게)
        def calc_skew(x):
            try:
                if len(x) > 2:
                    return skew(x)
            except:
                pass
            return 0
        
        result['ticket_skew'] = df.groupby('custid')['tot_amt'].apply(calc_skew)
        
        # 할인 민감도
        if 'dis_amt' in df.columns:
            df['discount_rate'] = df['dis_amt'] / (df['tot_amt'] + 1e-5)
            result['avg_discount_rate'] = df.groupby('custid')['discount_rate'].mean()
            result['max_discount_rate'] = df.groupby('custid')['discount_rate'].max()
            result['discount_seeker'] = (result['avg_discount_rate'] > 0.1).astype(int)
        
        # 명품 구매 비율
        if 'brd_nm' in df.columns:
            luxury_kw = ['명품', '수입명품', '부띠끄', '루이비통', '샤넬', '에르메스', '구찌']
            df['is_luxury'] = df['brd_nm'].apply(
                lambda x: int(any(k in str(x) for k in luxury_kw))
            )
            result['luxury_purchase_ratio'] = df.groupby('custid')['is_luxury'].mean()
        
        return result
    
    def _create_seasonal_response_features(self, df):
        """계절/이벤트 반응도 피처"""
        print("  🎄 계절/이벤트 반응도 피처 생성...")
        
        if 'sales_month' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        result = pd.DataFrame(index=df['custid'].unique())
        
        # 이벤트 시즌
        df['holiday_season'] = df['sales_month'].isin([11, 12, 1, 2]).astype(int)
        df['vacation_season'] = df['sales_month'].isin([1, 2, 7, 8]).astype(int)
        df['back_to_school'] = df['sales_month'].isin([2, 3, 8, 9]).astype(int)
        
        result['holiday_shopping_ratio'] = df.groupby('custid')['holiday_season'].mean()
        result['vacation_shopping_ratio'] = df.groupby('custid')['vacation_season'].mean()
        result['school_season_ratio'] = df.groupby('custid')['back_to_school'].mean()
        
        # 월별 다양성
        month_dist = df.groupby(['custid', 'sales_month']).size().unstack(fill_value=0)
        result['month_diversity'] = (month_dist > 0).sum(axis=1)
        
        # 이벤트 집중도
        total_purchase = df.groupby('custid').size()
        event_purchase = df[df['holiday_season'] == 1].groupby('custid').size()
        result['event_concentration'] = (event_purchase / total_purchase).fillna(0)
        
        return result
    
    def _create_family_composition_features(self, df):
        """가족 구성 추론 피처"""
        print("  👨‍👩‍👧‍👦 가족 구성 추론 피처 생성...")
        
        if 'pc_nm' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        result = pd.DataFrame(index=df['custid'].unique())
        
        # 아동 상품 구매
        kids_keywords = ['아동', '유아', '베이비', '키즈', '주니어', '완구', '문구', '학용품']
        df['kids_purchase'] = df['pc_nm'].apply(
            lambda x: int(any(k in str(x) for k in kids_keywords))
        )
        
        result['kids_purchase_count'] = df.groupby('custid')['kids_purchase'].sum()
        result['kids_purchase_ratio'] = df.groupby('custid')['kids_purchase'].mean()
        result['has_kids'] = (result['kids_purchase_count'] >= 2).astype(int)
        
        # 부부 쇼핑
        df['male_fashion'] = df['pc_nm'].str.contains('남성|신사', na=False, case=False).astype(int)
        df['female_fashion'] = df['pc_nm'].str.contains('여성|숙녀', na=False, case=False).astype(int)
        
        couple = df.groupby('custid').apply(
            lambda x: int((x['male_fashion'].sum() > 0) & (x['female_fashion'].sum() > 0))
        )
        result['couple_shopping'] = couple
        
        # 다세대 쇼핑
        df['senior_items'] = df['pc_nm'].str.contains('건강|실버|한복', na=False, case=False).astype(int)
        
        multi_gen = df.groupby('custid').apply(
            lambda x: int((x['kids_purchase'].sum() > 0) & (x['senior_items'].sum() > 0))
        )
        result['multi_generation'] = multi_gen
        
        return result
    
    def _create_digital_affinity_features(self, df):
        """디지털 친화도 피처"""
        print("  📱 디지털 친화도 피처 생성...")
        
        if 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        result = pd.DataFrame(index=df['custid'].unique())
        
        # 할부 사용
        if 'inst_mon' in df.columns:
            df['uses_installment'] = (df['inst_mon'] > 0).astype(int)
            df['long_installment'] = (df['inst_mon'] >= 6).astype(int)
            
            result['installment_usage_ratio'] = df.groupby('custid')['uses_installment'].mean()
            result['long_installment_ratio'] = df.groupby('custid')['long_installment'].mean()
            
            inst_users = df[df['inst_mon'] > 0]
            if not inst_users.empty:
                result['avg_installment_months'] = inst_users.groupby('custid')['inst_mon'].mean()
        
        # 수입품 선호
        if 'import_flg' in df.columns:
            result['import_preference'] = df.groupby('custid')['import_flg'].mean()
        
        # 다양성 추구
        if 'brd_nm' in df.columns:
            purchase_counts = df.groupby(['custid', 'brd_nm']).size().reset_index(name='count')
            avg_per_brand = purchase_counts.groupby('custid')['count'].mean()
            result['avg_purchase_per_brand'] = avg_per_brand
            result['variety_seeker'] = (avg_per_brand < 2).astype(int)
        
        return result
    
    def _create_statistical_features(self, df):
        """고급 통계 피처"""
        print("  📈 통계적 피처 생성...")
        
        if 'tot_amt' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        result = pd.DataFrame(index=df['custid'].unique())
        
        # 금액 기본 통계
        amt_agg = df.groupby('custid')['tot_amt'].agg([
            'mean', 'median', 'std', 'min', 'max'
        ])
        amt_agg.columns = ['amt_' + c for c in amt_agg.columns]
        
        # 분위수
        amt_q25 = df.groupby('custid')['tot_amt'].quantile(0.25).rename('amt_q25')
        amt_q75 = df.groupby('custid')['tot_amt'].quantile(0.75).rename('amt_q75')
        
        result = result.join(amt_agg)
        result = result.join(amt_q25)
        result = result.join(amt_q75)
        result['amt_iqr'] = result['amt_q75'] - result['amt_q25']
        
        # Skewness & Kurtosis (안전하게)
        def calc_skew(x):
            try:
                return skew(x) if len(x) > 2 else 0
            except:
                return 0
        
        def calc_kurt(x):
            try:
                return kurtosis(x) if len(x) > 3 else 0
            except:
                return 0
        
        result['amt_skew'] = df.groupby('custid')['tot_amt'].apply(calc_skew)
        result['amt_kurt'] = df.groupby('custid')['tot_amt'].apply(calc_kurt)
        
        # 구매 빈도
        if 'sales_day' in df.columns and 'sales_month' in df.columns:
            df_sorted = df.sort_values(['custid', 'sales_month', 'sales_day'])
            df_sorted['cumcount'] = df_sorted.groupby('custid').cumcount()
            
            freq = df_sorted.groupby('custid')['cumcount'].apply(
                lambda x: len(x) / (x.max() + 1) if x.max() > 0 else 0
            )
            result['purchase_frequency_days'] = freq
        
        # 시간 집중도
        if 'sales_time' in df.columns:
            hour_counts = df.groupby(['custid', df['sales_time'] // 100]).size().unstack(fill_value=0)
            time_conc = hour_counts.apply(
                lambda x: (x ** 2).sum() / (x.sum() ** 2) if x.sum() > 0 else 0,
                axis=1
            )
            result['time_concentration'] = time_conc
        
        return result


def main():
    """메인 실행 함수"""
    print("\n" + "="*60)
    print("🎯 연령 예측 모델 - 고도화 피처 생성")
    print("="*60 + "\n")
    
    # 데이터 로드
    print("📂 데이터 로딩...")
    train_tx = pd.read_csv("data/train_transactions.csv", encoding='cp949')
    test_tx = pd.read_csv("data/test_transactions.csv", encoding='cp949')
    
    train_agg = pd.read_csv("data/X_train_AGG.csv")
    test_agg = pd.read_csv("data/X_test_AGG.csv")
    
    train_w2v = pd.read_csv("data/X_train_W2V.csv")
    test_w2v = pd.read_csv("data/X_test_W2V.csv")
    
    y_train = pd.read_csv("data/y_train.csv")
    
    print(f"✅ Train 거래: {len(train_tx):,}건")
    print(f"✅ Test 거래: {len(test_tx):,}건")
    print(f"✅ Train 고객: {train_agg.shape[0]:,}명")
    print(f"✅ Test 고객: {test_agg.shape[0]:,}명")
    
    # 피처 생성
    engineer = AgeFeatureEngineer()
    
    print("\n" + "="*60)
    print("🔧 Train 피처 생성")
    print("="*60)
    train_features = engineer.create_age_specific_features(
        train_tx, train_agg, train_w2v
    )
    
    print("\n" + "="*60)
    print("🔧 Test 피처 생성")
    print("="*60)
    test_features = engineer.create_age_specific_features(
        test_tx, test_agg, test_w2v
    )
    
    # custid를 컬럼으로 변환
    train_features = train_features.reset_index()
    test_features = test_features.reset_index()
    
    # custid 컬럼명 확인 및 통일
    if 'index' in train_features.columns and 'custid' not in train_features.columns:
        train_features = train_features.rename(columns={'index': 'custid'})
    if 'index' in test_features.columns and 'custid' not in test_features.columns:
        test_features = test_features.rename(columns={'index': 'custid'})
    
    # 타겟 병합
    train_final = train_features.merge(
        y_train[['custid', 'age']], on='custid', how='inner'
    )
    
    # 컬럼 정렬 (train 기준)
    feature_cols = [c for c in train_final.columns if c not in ['custid', 'age']]
    
    # test에 없는 컬럼 추가
    for col in feature_cols:
        if col not in test_features.columns:
            test_features[col] = 0
    
    # 컬럼 순서 맞추기
    test_final = test_features[['custid'] + feature_cols]
    train_final = train_final[['custid'] + feature_cols + ['age']]
    
    # 결측치 및 무한대 처리
    train_final = train_final.fillna(0).replace([np.inf, -np.inf], 0)
    test_final = test_final.fillna(0).replace([np.inf, -np.inf], 0)
    
    # 저장
    print("\n" + "="*60)
    print("💾 결과 저장")
    print("="*60)
    
    train_final.to_csv("data/X_train_enhanced_claude1.csv", index=False, encoding='utf-8-sig')
    test_final.to_csv("data/X_test_enhanced_claude1.csv", index=False, encoding='utf-8-sig')
    
    print(f"\n✅ Train: {train_final.shape} → data/X_train_enhanced_claude.csv")
    print(f"✅ Test: {test_final.shape} → data/X_test_enhanced_claude.csv")
    print(f"\n📊 최종 피처 수: {len(feature_cols)}개")
    print("\n🎉 완료!")


if __name__ == "__main__":
    main()


🎯 연령 예측 모델 - 고도화 피처 생성

📂 데이터 로딩...
✅ Train 거래: 625,084건
✅ Test 거래: 414,955건
✅ Train 고객: 21,587명
✅ Test 고객: 14,380명

🔧 Train 피처 생성
🎯 연령 예측 특화 피처 생성 시작...
  📊 브랜드 연령 선호도 피처 생성...
  ⏰ 시간대 패턴 피처 생성...
  🏷️ 카테고리 조합 피처 생성...
  💰 소비 성향 피처 생성...
  🎄 계절/이벤트 반응도 피처 생성...
  👨‍👩‍👧‍👦 가족 구성 추론 피처 생성...
  📱 디지털 친화도 피처 생성...
  📈 통계적 피처 생성...
✅ 총 생성된 피처 수: 919개

🔧 Test 피처 생성
🎯 연령 예측 특화 피처 생성 시작...
  📊 브랜드 연령 선호도 피처 생성...
  ⏰ 시간대 패턴 피처 생성...
  🏷️ 카테고리 조합 피처 생성...
  💰 소비 성향 피처 생성...
  🎄 계절/이벤트 반응도 피처 생성...
  👨‍👩‍👧‍👦 가족 구성 추론 피처 생성...
  📱 디지털 친화도 피처 생성...
  📈 통계적 피처 생성...
✅ 총 생성된 피처 수: 919개

💾 결과 저장

✅ Train: (21587, 921) → data/X_train_enhanced_claude.csv
✅ Test: (14380, 920) → data/X_test_enhanced_claude.csv

📊 최종 피처 수: 919개

🎉 완료!


V3

In [35]:
"""
🎯 연령 예측 모델 - 고도화 피처 엔지니어링 + Target Encoding
=============================================================
- 입력:
    data/train_transactions.csv
    data/test_transactions.csv
    data/X_train_AGG.csv
    data/X_test_AGG.csv
    data/X_train_W2V.csv
    data/X_test_W2V.csv
    data/y_train.csv  (custid, age)

- 출력:
    data/X_train_enhanced.csv
    data/X_test_enhanced.csv

- 목적:
    1) AGG + W2V + 도메인 기반 FE
    2) 거래 범주형 데이터 Target Encoding(age, regression)
    3) Low-variance & Sparse feature 제거로 과적합 완화
"""

import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')


# =========================================================
# 1. Transaction Target Encoder (Regression, Leakage-free)
# =========================================================
class TransactionTargetEncoderReg:
    """
    거래 단위 범주형 컬럼을 나이(age) 기준으로 타겟 인코딩하고,
    고객(custid) 단위로 집계하는 회귀용 Target Encoding 모듈.
    
    - KFold를 custid 기준으로 나눠서 OOF 방식으로 학습 → 데이터 누수 방지
    - test는 전체 train으로 학습한 통계로만 인코딩
    """
    def __init__(
        self,
        cat_cols,
        target_col='age',
        id_col='custid',
        n_splits=5,
        min_samples=10,
        noise=0.01,
        max_cardinality=2000,
        random_state=42
    ):
        self.cat_cols = cat_cols
        self.target_col = target_col
        self.id_col = id_col
        self.n_splits = n_splits
        self.min_samples = min_samples
        self.noise = noise
        self.max_cardinality = max_cardinality
        self.random_state = random_state
        
        self.encoding_stats_ = {}   # col -> {'stats': DataFrame, 'global_mean': float}
        self.valid_cols_ = []       # 실제로 인코딩에 사용된 컬럼 목록

    def _encode_single_col_oof(self, tx, y_df, col):
        """
        단일 범주형 컬럼에 대해 OOF target encoding 수행.
        - tx : 거래 데이터 + target(age) merge된 상태
        - y_df : 고객 단위 타깃 (custid, age)
        """
        id_col = self.id_col
        target_col = self.target_col

        # Cardinality 체크 (너무 많은 카테고리는 스킵)
        nunique = tx[col].nunique()
        if nunique > self.max_cardinality:
            print(f"    ⚠️ {col}: cardinality={nunique} > {self.max_cardinality}, 스킵")
            return None

        print(f"    ▶ {col} (unique={nunique}) OOF Target Encoding 중...")

        # custid 단위로 KFold
        cust_ids = y_df[id_col].unique()
        kf = KFold(
            n_splits=self.n_splits,
            shuffle=True,
            random_state=self.random_state
        )

        # 결과 저장용
        te_series = pd.Series(index=cust_ids, dtype=float)

        for fold, (tr_idx, val_idx) in enumerate(kf.split(cust_ids)):
            train_ids = cust_ids[tr_idx]
            val_ids = cust_ids[val_idx]

            tr_tx = tx[tx[id_col].isin(train_ids)].copy()
            val_tx = tx[tx[id_col].isin(val_ids)].copy()

            # 범주형 전처리
            tr_tx[col] = tr_tx[col].fillna('UNKNOWN').astype(str)
            val_tx[col] = val_tx[col].fillna('UNKNOWN').astype(str)

            # train fold에서 category별 age 통계
            stats = tr_tx.groupby(col)[target_col].agg(['mean', 'count'])
            global_mean = tr_tx[target_col].mean()

            # 샘플 수 적은 카테고리는 global_mean 사용
            stats['te_value'] = np.where(
                stats['count'] >= self.min_samples,
                stats['mean'],
                global_mean
            )

            # validation fold에 매핑
            val_tx = val_tx.merge(
                stats[['te_value']],
                left_on=col,
                right_index=True,
                how='left'
            )
            val_tx['te_value'] = val_tx['te_value'].fillna(global_mean)

            # 노이즈 추가 (과적합 방지)
            if self.noise > 0:
                val_tx['te_value'] += np.random.normal(
                    0, self.noise, size=len(val_tx)
                )

            # 고객 단위 평균으로 집계
            cust_enc = val_tx.groupby(id_col)['te_value'].mean()
            te_series.loc[cust_enc.index] = cust_enc.values

        # 전체 train으로 다시 통계 계산 → test용
        tx_all = tx.copy()
        tx_all[col] = tx_all[col].fillna('UNKNOWN').astype(str)
        full_stats = tx_all.groupby(col)[target_col].agg(['mean', 'count'])
        full_global_mean = tx_all[target_col].mean()
        full_stats['te_value'] = np.where(
            full_stats['count'] >= self.min_samples,
            full_stats['mean'],
            full_global_mean
        )

        # 저장
        self.encoding_stats_[col] = {
            'stats': full_stats[['te_value']],
            'global_mean': full_global_mean
        }
        self.valid_cols_.append(col)

        return te_series

    def fit_transform(self, trans_train, y_train):
        """
        train 거래 데이터와 y_train(custid, age)을 이용해
        타겟 인코딩 피처를 만들고 (custid 단위), DataFrame으로 반환.
        """
        id_col = self.id_col
        target_col = self.target_col

        print("\n✨ [Target Encoding] 거래 범주형 → age 회귀용 인코딩 시작")

        # 타입 통일 (문자열)
        trans_train = trans_train.copy()
        y_train = y_train.copy()
        trans_train[id_col] = trans_train[id_col].astype(str)
        y_train[id_col] = y_train[id_col].astype(str)

        # 거래 데이터에 age 붙이기
        tx = trans_train.merge(
            y_train[[id_col, target_col]],
            on=id_col,
            how='inner'
        )

        te_frames = []
        for col in self.cat_cols:
            if col not in tx.columns:
                print(f"    ⚠️ {col}: 거래 데이터에 없음, 스킵")
                continue

            te_series = self._encode_single_col_oof(tx, y_train, col)
            if te_series is None:
                continue

            te_df = te_series.to_frame(name=f"te_{col}")
            te_frames.append(te_df)

        if not te_frames:
            print("    ⚠️ 유효한 범주형 컬럼이 없어 Target Encoding 결과가 비어있습니다.")
            return pd.DataFrame(index=y_train[id_col].unique())

        te_result = pd.concat(te_frames, axis=1)
        print(f"✨ Target Encoding 완료: {te_result.shape[1]}개 피처 생성")
        return te_result

    def transform(self, trans_test):
        """
        test 거래 데이터에 대해,
        train에서 학습한 인코딩 통계를 사용해 te_* 피처를 생성.
        """
        id_col = self.id_col
        trans_test = trans_test.copy()
        trans_test[id_col] = trans_test[id_col].astype(str)

        if not self.valid_cols_:
            print("⚠️ fit_transform이 먼저 실행되지 않아 Target Encoding 결과가 없습니다.")
            return pd.DataFrame(index=trans_test[id_col].unique())

        te_frames = []
        for col in self.valid_cols_:
            info = self.encoding_stats_[col]
            stats = info['stats']
            global_mean = info['global_mean']

            if col not in trans_test.columns:
                print(f"    ⚠️ {col}: test에 없음, 스킵")
                continue

            tx = trans_test[[id_col, col]].copy()
            tx[col] = tx[col].fillna('UNKNOWN').astype(str)

            tx = tx.merge(stats, left_on=col, right_index=True, how='left')
            tx['te_value'] = tx['te_value'].fillna(global_mean)

            cust_enc = tx.groupby(id_col)['te_value'].mean()
            te_frames.append(cust_enc.to_frame(f"te_{col}"))

        if not te_frames:
            return pd.DataFrame(index=trans_test[id_col].unique())

        te_result = pd.concat(te_frames, axis=1)
        return te_result


# =========================================================
# 2. AgeFeatureEngineer (AGG + W2V + Rule-based FE)
# =========================================================
class AgeFeatureEngineer:
    """연령 예측을 위한 고급 피처 엔지니어링 (AGG + W2V + 거래 기반 FE)"""
    
    def __init__(self):
        # 연령대별 선호 브랜드 (도메인 지식 기반, 필요시 튜닝 가능)
        self.young_brands = ['지오다노', '유니클로', '자라', 'H&M', '탑텐', '스파오', '에잇세컨즈']
        self.middle_brands = ['로가디스', '타임', '마리떼', '엠브로', '빈폴']
        self.senior_brands = ['한섬', '루이까또즈', '파크론', '머렐']
        
    def create_age_specific_features(self, trans_df, agg_df, w2v_df):
        """
        연령 예측을 위한 특화 피처 생성 (custid index)
        - agg_df, w2v_df: 이미 고객 단위 피처
        - trans_df: 거래 단위 (여기서 추가 rule-based 피처 생성)
        """
        print("🎯 연령 예측 특화 피처 생성 시작...")
        
        # 기본 통합 (AGG + W2V)
        base_features = self._merge_base_features(agg_df, w2v_df)  # index=custid
        
        # 거래 데이터 기반 rule-based 피처들
        brand_features      = self._create_brand_age_features(trans_df)
        time_features       = self._create_time_pattern_features(trans_df)
        category_features   = self._create_category_combination_features(trans_df)
        spending_features   = self._create_spending_behavior_features(trans_df)
        seasonal_features   = self._create_seasonal_response_features(trans_df)
        family_features     = self._create_family_composition_features(trans_df)
        digital_features    = self._create_digital_affinity_features(trans_df)
        statistical_features= self._create_statistical_features(trans_df)
        
        feature_list = [
            base_features,
            brand_features,
            time_features,
            category_features,
            spending_features,
            seasonal_features,
            family_features,
            digital_features,
            statistical_features
        ]
        
        # None/빈 DF 제거 후 index(custid) 기준으로 join
        valid_features = [f for f in feature_list if f is not None and not f.empty]
        
        result = valid_features[0]
        for feat in valid_features[1:]:
            result = result.join(feat, how='outer')
        
        print(f"✅ 도메인 + AGG + W2V 피처 수: {result.shape[1]}개")
        return result  # index=custid
    
    def _merge_base_features(self, agg_df, w2v_df):
        """AGG + W2V 통합 (custid 기준)"""
        agg_df = agg_df.copy()
        w2v_df = w2v_df.copy()
        agg_df['custid'] = agg_df['custid'].astype(str)
        w2v_df['custid'] = w2v_df['custid'].astype(str)
        
        base = agg_df.merge(w2v_df, on='custid', how='outer')
        return base.set_index('custid')
    
    # ------------------- 이하 rule-based FE는 거의 그대로 -------------------
    def _create_brand_age_features(self, df):
        print("  📊 브랜드 연령 선호도 피처 생성...")
        if 'brd_nm' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        df['custid'] = df['custid'].astype(str)
        result = pd.DataFrame(index=df['custid'].unique())
        
        brand_amt = df.groupby(['custid', 'brd_nm'])['tot_amt'].sum().unstack(fill_value=0)
        total_amt = brand_amt.sum(axis=1)
        
        young_cols = [b for b in self.young_brands if b in brand_amt.columns]
        middle_cols = [b for b in self.middle_brands if b in brand_amt.columns]
        senior_cols = [b for b in self.senior_brands if b in brand_amt.columns]
        
        if young_cols:
            result['brand_young_ratio'] = brand_amt[young_cols].sum(axis=1) / (total_amt + 1e-5)
        if middle_cols:
            result['brand_middle_ratio'] = brand_amt[middle_cols].sum(axis=1) / (total_amt + 1e-5)
        if senior_cols:
            result['brand_senior_ratio'] = brand_amt[senior_cols].sum(axis=1) / (total_amt + 1e-5)
        
        result['brand_diversity'] = df.groupby('custid')['brd_nm'].nunique()
        brand_conc = df.groupby('custid')['brd_nm'].apply(
            lambda x: (x.value_counts().iloc[0] / len(x)) if len(x) > 0 else 0
        )
        result['brand_concentration'] = brand_conc
        
        return result
    
    def _create_time_pattern_features(self, df):
        print("  ⏰ 시간대 패턴 피처 생성...")
        if 'sales_time' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        df['custid'] = df['custid'].astype(str)
        df['hour'] = (df['sales_time'] // 100).clip(0, 23)
        result = pd.DataFrame(index=df['custid'].unique())
        
        if 'sales_dayofweek' in df.columns:
            weekday_map = {'월': 0, '화': 1, '수': 2, '목': 3, '금': 4, '토': 5, '일': 6}
            df['dow_num'] = pd.to_numeric(df['sales_dayofweek'].map(weekday_map), errors='coerce')
            df['dow_num'] = df['dow_num'].fillna(df['sales_dayofweek'])
            df['is_weekday'] = df['dow_num'] <= 4
        
        df['early_morning'] = (df['hour'] < 10).astype(int)
        df['lunch_time'] = df['hour'].between(11, 14).astype(int)
        df['after_school'] = df['hour'].between(16, 19).astype(int)
        df['evening'] = df['hour'].between(19, 21).astype(int)
        df['late_night'] = (df['hour'] >= 21).astype(int)
        
        if 'is_weekday' in df.columns:
            df['weekday_lunch'] = (df['is_weekday'] & (df['lunch_time'] == 1)).astype(int)
            df['weekday_morning'] = (df['is_weekday'] & (df['early_morning'] == 1)).astype(int)
            df['weekday_afterschool'] = (df['is_weekday'] & (df['after_school'] == 1)).astype(int)
            
            result['weekday_lunch_ratio'] = df.groupby('custid')['weekday_lunch'].mean()
            result['weekday_morning_ratio'] = df.groupby('custid')['weekday_morning'].mean()
            result['weekday_afterschool_ratio'] = df.groupby('custid')['weekday_afterschool'].mean()
        
        for col in ['early_morning', 'lunch_time', 'after_school', 'evening', 'late_night']:
            result[f'{col}_ratio'] = df.groupby('custid')[col].mean()
        
        result['hour_std'] = df.groupby('custid')['hour'].std()
        result['hour_range'] = df.groupby('custid')['hour'].apply(lambda x: x.max() - x.min())
        
        return result
    
    def _create_category_combination_features(self, df):
        print("  🏷️ 카테고리 조합 피처 생성...")
        if 'pc_nm' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        df['custid'] = df['custid'].astype(str)
        result = pd.DataFrame(index=df['custid'].unique())
        
        df['kids_item'] = df['pc_nm'].str.contains('아동|유아|베이비|키즈', na=False, case=False).astype(int)
        kid_buyers = df[df['kids_item'] == 1]['custid'].unique()
        result['has_kids_signal'] = result.index.isin(kid_buyers).astype(int)
        
        df['luxury'] = df['pc_nm'].str.contains('명품|보석|주얼리', na=False, case=False).astype(int)
        df['golf'] = df['pc_nm'].str.contains('골프', na=False, case=False).astype(int)
        luxury_golf = df.groupby('custid').apply(
            lambda x: int((x['luxury'].sum() > 0) & (x['golf'].sum() > 0))
        )
        result['luxury_golf_combo'] = luxury_golf
        
        df['cosmetics'] = df['pc_nm'].str.contains('화장품|뷰티|메이크업', na=False, case=False).astype(int)
        df['accessories'] = df['pc_nm'].str.contains('패션잡화|악세사리|핸드백', na=False, case=False).astype(int)
        young_female = df.groupby('custid').apply(
            lambda x: int((x['cosmetics'].sum() > 0) & (x['accessories'].sum() > 0))
        )
        result['young_female_combo'] = young_female
        
        if 'corner_nm' in df.columns:
            df['food'] = df['corner_nm'].str.contains('식품', na=False, case=False).astype(int)
            df['kitchen'] = df['corner_nm'].str.contains('주방|취사', na=False, case=False).astype(int)
            housewife = df.groupby('custid').apply(
                lambda x: int((x['food'].sum() > 0) & (x['kitchen'].sum() > 0))
            )
            result['housewife_combo'] = housewife
        
        return result
    
    def _create_spending_behavior_features(self, df):
        print("  💰 소비 성향 피처 생성...")
        if 'tot_amt' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        df['custid'] = df['custid'].astype(str)
        result = pd.DataFrame(index=df['custid'].unique())
        
        df['price_tier'] = pd.cut(
            df['tot_amt'],
            bins=[-np.inf, 50000, 100000, 200000, 500000, np.inf],
            labels=['very_low', 'low', 'mid', 'high', 'premium']
        )
        price_dist = pd.crosstab(df['custid'], df['price_tier'], normalize='index')
        for col in price_dist.columns:
            result[f'price_{col}_ratio'] = price_dist[col]
        
        cust_stats = df.groupby('custid')['tot_amt'].agg(['mean', 'std', 'median'])
        result['ticket_cv'] = cust_stats['std'] / (cust_stats['mean'] + 1e-5)
        
        def calc_skew(x):
            try:
                return skew(x) if len(x) > 2 else 0
            except:
                return 0
        result['ticket_skew'] = df.groupby('custid')['tot_amt'].apply(calc_skew)
        
        if 'dis_amt' in df.columns:
            df['discount_rate'] = df['dis_amt'] / (df['tot_amt'] + 1e-5)
            result['avg_discount_rate'] = df.groupby('custid')['discount_rate'].mean()
            result['max_discount_rate'] = df.groupby('custid')['discount_rate'].max()
            result['discount_seeker'] = (result['avg_discount_rate'] > 0.1).astype(int)
        
        if 'brd_nm' in df.columns:
            luxury_kw = ['명품', '수입명품', '부띠끄', '루이비통', '샤넬', '에르메스', '구찌']
            df['is_luxury'] = df['brd_nm'].apply(
                lambda x: int(any(k in str(x) for k in luxury_kw))
            )
            result['luxury_purchase_ratio'] = df.groupby('custid')['is_luxury'].mean()
        
        return result
    
    def _create_seasonal_response_features(self, df):
        print("  🎄 계절/이벤트 반응도 피처 생성...")
        if 'sales_month' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        df['custid'] = df['custid'].astype(str)
        result = pd.DataFrame(index=df['custid'].unique())
        
        df['holiday_season'] = df['sales_month'].isin([11, 12, 1, 2]).astype(int)
        df['vacation_season'] = df['sales_month'].isin([1, 2, 7, 8]).astype(int)
        df['back_to_school'] = df['sales_month'].isin([2, 3, 8, 9]).astype(int)
        
        result['holiday_shopping_ratio'] = df.groupby('custid')['holiday_season'].mean()
        result['vacation_shopping_ratio'] = df.groupby('custid')['vacation_season'].mean()
        result['school_season_ratio'] = df.groupby('custid')['back_to_school'].mean()
        
        month_dist = df.groupby(['custid', 'sales_month']).size().unstack(fill_value=0)
        result['month_diversity'] = (month_dist > 0).sum(axis=1)
        
        total_purchase = df.groupby('custid').size()
        event_purchase = df[df['holiday_season'] == 1].groupby('custid').size()
        result['event_concentration'] = (event_purchase / total_purchase).fillna(0)
        
        return result
    
    def _create_family_composition_features(self, df):
        print("  👨‍👩‍👧‍👦 가족 구성 추론 피처 생성...")
        if 'pc_nm' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        df['custid'] = df['custid'].astype(str)
        result = pd.DataFrame(index=df['custid'].unique())
        
        kids_keywords = ['아동', '유아', '베이비', '키즈', '주니어', '완구', '문구', '학용품']
        df['kids_purchase'] = df['pc_nm'].apply(
            lambda x: int(any(k in str(x) for k in kids_keywords))
        )
        
        result['kids_purchase_count'] = df.groupby('custid')['kids_purchase'].sum()
        result['kids_purchase_ratio'] = df.groupby('custid')['kids_purchase'].mean()
        result['has_kids'] = (result['kids_purchase_count'] >= 2).astype(int)
        
        df['male_fashion'] = df['pc_nm'].str.contains('남성|신사', na=False, case=False).astype(int)
        df['female_fashion'] = df['pc_nm'].str.contains('여성|숙녀', na=False, case=False).astype(int)
        
        couple = df.groupby('custid').apply(
            lambda x: int((x['male_fashion'].sum() > 0) & (x['female_fashion'].sum() > 0))
        )
        result['couple_shopping'] = couple
        
        df['senior_items'] = df['pc_nm'].str.contains('건강|실버|한복', na=False, case=False).astype(int)
        multi_gen = df.groupby('custid').apply(
            lambda x: int((x['kids_purchase'].sum() > 0) & (x['senior_items'].sum() > 0))
        )
        result['multi_generation'] = multi_gen
        
        return result
    
    def _create_digital_affinity_features(self, df):
        print("  📱 디지털 친화도 피처 생성...")
        if 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        df['custid'] = df['custid'].astype(str)
        result = pd.DataFrame(index=df['custid'].unique())
        
        if 'inst_mon' in df.columns:
            df['uses_installment'] = (df['inst_mon'] > 0).astype(int)
            df['long_installment'] = (df['inst_mon'] >= 6).astype(int)
            
            result['installment_usage_ratio'] = df.groupby('custid')['uses_installment'].mean()
            result['long_installment_ratio'] = df.groupby('custid')['long_installment'].mean()
            
            inst_users = df[df['inst_mon'] > 0]
            if not inst_users.empty:
                result['avg_installment_months'] = inst_users.groupby('custid')['inst_mon'].mean()
        
        if 'import_flg' in df.columns:
            result['import_preference'] = df.groupby('custid')['import_flg'].mean()
        
        if 'brd_nm' in df.columns:
            purchase_counts = df.groupby(['custid', 'brd_nm']).size().reset_index(name='count')
            avg_per_brand = purchase_counts.groupby('custid')['count'].mean()
            result['avg_purchase_per_brand'] = avg_per_brand
            result['variety_seeker'] = (avg_per_brand < 2).astype(int)
        
        return result
    
    def _create_statistical_features(self, df):
        print("  📈 통계적 피처 생성...")
        if 'tot_amt' not in df.columns or 'custid' not in df.columns:
            return pd.DataFrame()
        
        df = df.copy()
        df['custid'] = df['custid'].astype(str)
        result = pd.DataFrame(index=df['custid'].unique())
        
        amt_agg = df.groupby('custid')['tot_amt'].agg(['mean', 'median', 'std', 'min', 'max'])
        amt_agg.columns = ['amt_' + c for c in amt_agg.columns]
        
        amt_q25 = df.groupby('custid')['tot_amt'].quantile(0.25).rename('amt_q25')
        amt_q75 = df.groupby('custid')['tot_amt'].quantile(0.75).rename('amt_q75')
        
        result = result.join(amt_agg)
        result = result.join(amt_q25)
        result = result.join(amt_q75)
        result['amt_iqr'] = result['amt_q75'] - result['amt_q25']
        
        def calc_skew(x):
            try:
                return skew(x) if len(x) > 2 else 0
            except:
                return 0
        def calc_kurt(x):
            try:
                return kurtosis(x) if len(x) > 3 else 0
            except:
                return 0
        
        result['amt_skew'] = df.groupby('custid')['tot_amt'].apply(calc_skew)
        result['amt_kurt'] = df.groupby('custid')['tot_amt'].apply(calc_kurt)
        
        if 'sales_day' in df.columns and 'sales_month' in df.columns:
            df_sorted = df.sort_values(['custid', 'sales_month', 'sales_day'])
            df_sorted['cumcount'] = df_sorted.groupby('custid').cumcount()
            freq = df_sorted.groupby('custid')['cumcount'].apply(
                lambda x: len(x) / (x.max() + 1) if x.max() > 0 else 0
            )
            result['purchase_frequency_days'] = freq
        
        if 'sales_time' in df.columns:
            hour_counts = df.groupby(['custid', df['sales_time'] // 100]).size().unstack(fill_value=0)
            time_conc = hour_counts.apply(
                lambda x: (x ** 2).sum() / (x.sum() ** 2) if x.sum() > 0 else 0,
                axis=1
            )
            result['time_concentration'] = time_conc
        
        return result


# =========================================================
# 3. Feature Cleaning (Low-variance + Sparse 제거)
# =========================================================
def clean_features(X_train, X_test, low_var_th=1e-6, sparse_th=0.003):
    """
    - 분산이 거의 없는 피처 제거 (low_var_th 이하)
    - 대부분이 0인 Sparse 피처 제거 (non-zero 비율 sparse_th 미만)
    """
    print("\n🧹 [Feature Cleaning] Low-variance & Sparse 피처 제거 중...")
    
    variances = X_train.var()
    low_var_cols = variances[variances < low_var_th].index.tolist()
    
    non_zero_ratio = (X_train != 0).mean()
    sparse_cols = non_zero_ratio[non_zero_ratio < sparse_th].index.tolist()
    
    drop_cols = sorted(set(low_var_cols + sparse_cols))
    
    print(f"   📉 Low-variance 제거: {len(low_var_cols)}개")
    print(f"   📉 Sparse 제거: {len(sparse_cols)}개 (threshold={sparse_th})")
    print(f"   🗑️ 총 제거 피처 수: {len(drop_cols)}개")
    
    X_train_clean = X_train.drop(columns=drop_cols, errors='ignore')
    X_test_clean = X_test.drop(columns=drop_cols, errors='ignore')
    
    print(f"   ✅ 정리 후 Train 피처 수: {X_train_clean.shape[1]}개")
    return X_train_clean, X_test_clean


# =========================================================
# 4. Main Pipeline
# =========================================================
def main():
    print("\n" + "="*70)
    print("🎯 연령 예측 모델 - 고도화 FE + Target Encoding (Regression/RMSE)")
    print("="*70 + "\n")
    
    # ---------- 데이터 로드 ----------
    print("📂 데이터 로딩...")
    train_tx = pd.read_csv("data/train_transactions.csv", encoding='cp949')
    test_tx  = pd.read_csv("data/test_transactions.csv",  encoding='cp949')
    
    train_agg = pd.read_csv("data/X_train_AGG.csv")
    test_agg  = pd.read_csv("data/X_test_AGG.csv")
    
    train_w2v = pd.read_csv("data/X_train_W2V.csv")
    test_w2v  = pd.read_csv("data/X_test_W2V.csv")
    
    y_train   = pd.read_csv("data/y_train.csv")
    
    print(f"✅ Train 거래: {len(train_tx):,}건")
    print(f"✅ Test 거래 : {len(test_tx):,}건")
    print(f"✅ Train 고객: {train_agg.shape[0]:,}명")
    print(f"✅ Test 고객 : {test_agg.shape[0]:,}명")
    
    # custid 타입 통일 (문자열로)
    for df in [train_tx, test_tx, train_agg, test_agg, train_w2v, test_w2v, y_train]:
        if 'custid' in df.columns:
            df['custid'] = df['custid'].astype(str)
    
    # ---------- 1) AGG + W2V + 도메인 FE ----------
    engineer = AgeFeatureEngineer()
    
    print("\n" + "="*60)
    print("🔧 Train 피처 생성 (AGG + W2V + 도메인 FE)")
    print("="*60)
    train_features = engineer.create_age_specific_features(
        train_tx, train_agg, train_w2v
    )
    
    print("\n" + "="*60)
    print("🔧 Test 피처 생성 (AGG + W2V + 도메인 FE)")
    print("="*60)
    test_features = engineer.create_age_specific_features(
        test_tx, test_agg, test_w2v
    )
    
    # ---------- 2) Target Encoding (거래 범주형 → age) ----------
    te_cat_cols = [col for col in ['brd_nm', 'corner_nm', 'pc_nm', 'part_nm', 'team_nm', 'buyer_nm', 'str_nm']
                   if col in train_tx.columns]
    
    te_encoder = TransactionTargetEncoderReg(
        cat_cols=te_cat_cols,
        target_col='age',
        id_col='custid',
        n_splits=5,
        min_samples=10,
        noise=0.01,
        max_cardinality=2000,
        random_state=42
    )
    
    te_train = te_encoder.fit_transform(train_tx, y_train)
    te_test  = te_encoder.transform(test_tx)
    
    # ---------- 3) 모든 피처 통합 ----------
    print("\n🔗 AGG+W2V+도메인 FE + Target Encoding 통합 중...")
    # index=custid 유지
    full_train = train_features.join(te_train, how='left')
    full_test  = test_features.join(te_test,  how='left')
    
    # 결측/무한대 처리
    full_train = full_train.fillna(0).replace([np.inf, -np.inf], 0)
    full_test  = full_test.fillna(0).replace([np.inf, -np.inf], 0)
    
    print(f"   👉 통합 후 Train 피처 수: {full_train.shape[1]}개")
    print(f"   👉 통합 후 Test 피처 수 : {full_test.shape[1]}개")
    
    # ---------- 4) Feature Cleaning ----------
    X_train_clean, X_test_clean = clean_features(full_train, full_test,
                                                 low_var_th=1e-6, sparse_th=0.003)
    
    # ---------- 5) custid 컬럼 복원 + y 병합 ----------
    X_train_clean = X_train_clean.reset_index().rename(columns={'index': 'custid'})
    X_test_clean  = X_test_clean.reset_index().rename(columns={'index': 'custid'})
    
    # 혹시 custid가 컬럼에 없으면 다시 이름 맞추기
    if 'custid' not in X_train_clean.columns and 'index' in X_train_clean.columns:
        X_train_clean = X_train_clean.rename(columns={'index': 'custid'})
    if 'custid' not in X_test_clean.columns and 'index' in X_test_clean.columns:
        X_test_clean = X_test_clean.rename(columns={'index': 'custid'})
    
    train_final = X_train_clean.merge(
        y_train[['custid', 'age']],
        on='custid',
        how='inner'
    )
    
    feature_cols = [c for c in train_final.columns if c not in ['custid', 'age']]
    test_final = X_test_clean[['custid'] + feature_cols].copy()
    train_final = train_final[['custid'] + feature_cols + ['age']]
    
    # 최종 결측/무한대 처리
    train_final = train_final.fillna(0).replace([np.inf, -np.inf], 0)
    test_final  = test_final.fillna(0).replace([np.inf, -np.inf], 0)
    
    # ---------- 6) 저장 ----------
    print("\n" + "="*60)
    print("💾 결과 저장")
    print("="*60)
    
    train_final.to_csv("data/X_train_enhanced_gpt2.csv", index=False, encoding='utf-8-sig')
    test_final.to_csv("data/X_test_enhanced_gpt2.csv",  index=False, encoding='utf-8-sig')
    
    print(f"\n✅ Train: {train_final.shape} → data/X_train_enhanced_gpt.csv")
    print(f"✅ Test : {test_final.shape}  → data/X_test_enhanced-gpt.csv")
    print(f"\n📊 최종 피처 수 (모델 입력 X): {len(feature_cols)}개")
    print("\n🎉 완료! (이제 AutoGluon에서 regression + rmse로 학습하면 됨)")


if __name__ == "__main__":
    main()



🎯 연령 예측 모델 - 고도화 FE + Target Encoding (Regression/RMSE)

📂 데이터 로딩...
✅ Train 거래: 625,084건
✅ Test 거래 : 414,955건
✅ Train 고객: 21,587명
✅ Test 고객 : 14,380명

🔧 Train 피처 생성 (AGG + W2V + 도메인 FE)
🎯 연령 예측 특화 피처 생성 시작...
  📊 브랜드 연령 선호도 피처 생성...
  ⏰ 시간대 패턴 피처 생성...
  🏷️ 카테고리 조합 피처 생성...
  💰 소비 성향 피처 생성...
  🎄 계절/이벤트 반응도 피처 생성...
  👨‍👩‍👧‍👦 가족 구성 추론 피처 생성...
  📱 디지털 친화도 피처 생성...
  📈 통계적 피처 생성...
✅ 도메인 + AGG + W2V 피처 수: 919개

🔧 Test 피처 생성 (AGG + W2V + 도메인 FE)
🎯 연령 예측 특화 피처 생성 시작...
  📊 브랜드 연령 선호도 피처 생성...
  ⏰ 시간대 패턴 피처 생성...
  🏷️ 카테고리 조합 피처 생성...
  💰 소비 성향 피처 생성...
  🎄 계절/이벤트 반응도 피처 생성...
  👨‍👩‍👧‍👦 가족 구성 추론 피처 생성...
  📱 디지털 친화도 피처 생성...
  📈 통계적 피처 생성...
✅ 도메인 + AGG + W2V 피처 수: 919개

✨ [Target Encoding] 거래 범주형 → age 회귀용 인코딩 시작
    ▶ brd_nm (unique=1848) OOF Target Encoding 중...
    ▶ corner_nm (unique=308) OOF Target Encoding 중...
    ▶ pc_nm (unique=77) OOF Target Encoding 중...
    ▶ part_nm (unique=30) OOF Target Encoding 중...
    ▶ team_nm (unique=4) OOF Target Encoding 중...
    ▶ buyer_nm (unique

V4 claude2